# Pandas Notebook 04: Transformations and feature engineering

This notebook shows how to transform columns and create new features for analytics and beginner data science workflows.

## Notebook objectives

By the end, you should be able to create new columns, use vectorized operations, work with `.str` and `.dt`, use `map()`, `replace()`, `apply()`, `np.where()`, `cut()`, and `qcut()`, and build useful analysis-ready features.

### Transforming vs engineering

- **Transforming** changes an existing column into a more useful form.
- **Feature engineering** creates a new column that adds useful information for analysis.

We often do both together.

In [ ]:
import pandas as pd
import numpy as np

## Dataset

We will use a local housing-style CSV stored in `sample_data`. It includes prices, sizes, dates, text, and categories, so it works well for feature engineering examples.

In [ ]:
csv_path = "sample_data/housing_listings_sample.csv"
housing = pd.read_csv(csv_path)
housing["listing_date"] = pd.to_datetime(housing["listing_date"])

housing

## 1-3. Creating new columns, vectorized operations, and arithmetic between columns

Vectorized operations work on full columns at once. They are usually preferred over loops because they are simpler and faster.

These operations make it easy to create new columns such as ratios, totals, and scaled values.

In [ ]:
housing_math = housing.copy()
housing_math["price_in_thousands"] = housing_math["price"] / 1000
housing_math["price_plus_5pct"] = housing_math["price"] * 1.05
housing_math["price_per_sqft"] = (housing_math["price"] / housing_math["size_sqft"]).round(2)
housing_math["total_rooms"] = housing_math["bedrooms"] + housing_math["bathrooms"]

housing_math[["listing_id", "price", "price_in_thousands", "price_plus_5pct", "size_sqft", "price_per_sqft", "total_rooms"]]

## 4. String methods with `.str`

Use `.str` when you want to clean or transform text columns. This is great for titles, labels, city names, and categories.

In [ ]:
housing_text = housing.copy()
housing_text["listing_title_clean"] = housing_text["listing_title"].str.strip().str.title()
housing_text["neighborhood_clean"] = housing_text["neighborhood"].str.replace("-", " ", regex=False).str.title()
housing_text["city_slug"] = housing_text["city"].str.lower().str.replace(" ", "_", regex=False)
housing_text["title_has_family"] = housing_text["listing_title"].str.strip().str.lower().str.contains("family")

housing_text[["listing_title", "listing_title_clean", "neighborhood", "neighborhood_clean", "city_slug", "title_has_family"]]

## 5. Datetime accessors with `.dt`

After a column has been converted to datetime, `.dt` helps us pull out useful parts such as month, year, quarter, and weekday.

In [ ]:
housing_dates = housing.copy()
housing_dates["listing_year"] = housing_dates["listing_date"].dt.year
housing_dates["listing_month"] = housing_dates["listing_date"].dt.month_name()
housing_dates["listing_quarter"] = housing_dates["listing_date"].dt.quarter
housing_dates["listing_weekday"] = housing_dates["listing_date"].dt.day_name()

housing_dates[["listing_id", "listing_date", "listing_year", "listing_month", "listing_quarter", "listing_weekday"]]

## 6. `map()` and `replace()`

`map()` is helpful for one-to-one mappings, while `replace()` is useful when swapping specific values. Both are common in categorical feature engineering.

In [ ]:
housing_map = housing.copy()
housing_map["energy_score"] = housing_map["energy_rating"].map({"A": 3, "B": 2, "C": 1})
housing_map["furnished_flag"] = housing_map["furnished"].map({"yes": 1, "no": 0})
housing_map["property_type_short"] = housing_map["property_type"].replace({"Apartment": "apt", "House": "house", "Studio": "studio"})

housing_map[["listing_id", "energy_rating", "energy_score", "furnished", "furnished_flag", "property_type", "property_type_short"]]

## 7-8. `apply()` and conditional feature creation with NumPy

`apply()` runs a function on each value or row. It is useful for custom logic, but if a vectorized option exists, that is usually the better first choice.

For simple conditional features, `np.where()` is often cleaner than row-by-row logic.

In [ ]:
housing_apply = housing.copy()

# Vectorized solution
housing_apply["price_per_sqft_vectorized"] = (housing_apply["price"] / housing_apply["size_sqft"]).round(2)

# Same idea with apply
housing_apply["price_per_sqft_apply"] = housing_apply.apply(
    lambda row: round(row["price"] / row["size_sqft"], 2),
    axis=1,
)

# A custom label with apply
housing_apply["home_size_label"] = housing_apply["size_sqft"].apply(
    lambda sqft: "large" if sqft >= 1400 else "medium" if sqft >= 900 else "small"
)

# A simple conditional feature with NumPy
housing_apply["is_family_home"] = np.where(housing_apply["bedrooms"] >= 3, "yes", "no")
housing_apply["listing_segment"] = np.where(
    (housing_apply["price"] >= 200000) | (housing_apply["size_sqft"] >= 1500),
    "premium",
    "standard",
)

housing_apply[["listing_id", "price_per_sqft_vectorized", "price_per_sqft_apply", "home_size_label", "is_family_home", "listing_segment"]]

## 9. Binning variables with `cut()` and `qcut()`

`cut()` uses bins you define. `qcut()` creates groups with roughly similar numbers of rows. Both are useful for turning continuous numbers into categories.

In [ ]:
housing_bins = housing.copy()
housing_bins["size_band"] = pd.cut(housing_bins["size_sqft"], bins=[0, 700, 1200, 2000], labels=["small", "medium", "large"])
housing_bins["price_tier"] = pd.qcut(housing_bins["price"], q=3, labels=["budget", "mid", "premium"])

housing_bins[["listing_id", "price", "size_sqft", "size_band", "price_tier"]]

## 10. Useful feature engineering examples for analysis

A good feature should help answer a question more easily. In housing data, helpful features often describe value, time, category, or household suitability.

In [ ]:
features_df = housing.copy()
features_df["listing_title_clean"] = features_df["listing_title"].str.strip().str.title()
features_df["price_per_sqft"] = (features_df["price"] / features_df["size_sqft"]).round(2)
features_df["listing_month"] = features_df["listing_date"].dt.month_name()
features_df["energy_score"] = features_df["energy_rating"].map({"A": 3, "B": 2, "C": 1})
features_df["furnished_flag"] = features_df["furnished"].map({"yes": 1, "no": 0})
features_df["is_family_home"] = np.where(features_df["bedrooms"] >= 3, 1, 0)
features_df["price_tier"] = pd.qcut(features_df["price"], q=3, labels=["budget", "mid", "premium"])
features_df["value_label"] = np.where(features_df["price_per_sqft"] < features_df["price_per_sqft"].median(), "better_value", "higher_cost")

features_df[["listing_id", "city", "property_type", "price_per_sqft", "listing_month", "energy_score", "furnished_flag", "is_family_home", "price_tier", "value_label"]]

## Practice exercises

Try these with `housing`.

1. Create a `price_per_bedroom` column.
2. Create a cleaned version of `listing_title`.
3. Create a `listing_month_number` column from `listing_date`.
4. Map `energy_rating` to a numeric score.
5. Use `np.where()` to label listings as `"large"` or `"not_large"` based on `size_sqft`.
6. Use `cut()` or `qcut()` to create a grouped price column.

In [ ]:
# Write your practice answers here.

## Mini challenge

Prepare a new DataFrame for later analysis.

Create at least these features:

- `price_per_sqft`
- `listing_month`
- `energy_score`
- `is_family_home`
- `price_tier`
- one extra feature of your own choice

Then answer:

- Which listings are in the `premium` tier?
- Which listings look like better value?
- Which city seems to have more family-sized homes in this sample?

In [ ]:
# Write your mini challenge solution here.

## When to use each method

- Use column assignment for straightforward new columns.
- Use vectorized arithmetic for fast column-wide math.
- Use `.str` for text work.
- Use `.dt` for date parts after datetime conversion.
- Use `map()` for one-to-one category mappings.
- Use `replace()` for targeted value swaps.
- Use `apply()` for custom logic when vectorized methods are not a good fit.
- Use `np.where()` for simple conditional labels.
- Use `cut()` for custom bins and `qcut()` for roughly equal-sized groups.

## Key takeaways

- transformations improve existing columns, while feature engineering creates new analytical columns
- vectorized methods are usually the best first choice in pandas
- text, dates, mappings, arithmetic, and conditions all help create useful features
- `apply()` is helpful, but it should not replace simpler vectorized solutions
- binned features can make reporting and analysis easier